# Opening-range results grouped by prior 10-day ADR

Read the completed **all-ticker** OR trade ledger from Drive. Do not rerun OR signals, fills or exits. Define ticker ADR as the arithmetic mean over the **ten prior completed regular sessions** of `100 × (high − low) / open`, matching the ARM 10-day ADR convention. The trade date's own range is never included. A regular session must have 9:30 and 15:59 one-minute bars and at least 350 distinct bars. Prior sessions must be the ten previous QQQ data dates, so missing stock days produce an unknown ADR rather than a stale value.

Primary subset: filled **shorts**, QQQ opening gap at most ±1%, ticker opening gap below −1%, 1-minute breakout confirmation, 1% stop / 2% target, November 4–5 excluded in original ledger. Keep QQQ at-entry with/against and before-noon/noon-onward splits. ADR bins are **<3%, 3–<4%, 4–<5%, 5–<7%, ≥7%**. Also show history group, stock price group, date concentration, and October 12/14/16 sensitivity. Every outcome table has wins/losses/neutrals and R. This is a descriptive subgroup analysis; no threshold is selected from these results.

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd, numpy as np, re
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
ROOT=next((p for p in [BASE/'backtesting',BASE/'Backtesting'] if p.is_dir()),None)
LEDGER=BASE/'Opening_Range_All_Tickers_QQQ_At_Entry'/'all_filled_trades_with_filters.csv'
OUT=BASE/'Opening_Range_All_Tickers_ADR_Study'; OUT.mkdir(exist_ok=True)
if ROOT is None: raise FileNotFoundError('Backtesting folder not found')
if not LEDGER.exists(): raise FileNotFoundError(LEDGER)
trades=pd.read_csv(LEDGER,dtype={'ticker':str,'date':str})
needed={'ticker','date','R','result','side','gap_pass','entry_window','qqq_vs_trade','history_group','price_group'}
if needed-set(trades): raise ValueError('Ledger missing: '+str(sorted(needed-set(trades))))
assert trades.result.isin(['win','loss','neutral']).all()
trades['gap_pass']=trades.gap_pass.astype(str).str.lower().eq('true')
primary=trades[trades.side.eq('short')&trades.gap_pass&trades.qqq_vs_trade.isin(['with','against'])].copy()
if primary.empty:raise RuntimeError('No qualifying short trades')
print('Qualifying short trades:',len(primary),'| tickers:',primary.ticker.nunique(),'| dates:',primary.date.nunique())

In [ ]:
wanted=set(primary.ticker)|{'QQQ'}
files={}
for p in ROOT.rglob('*_1m_*.csv'):
    m=re.match(r'(.+)_1m_\d{4}-\d{2}-\d{2}',p.stem)
    if m and m.group(1).upper() in wanted: files.setdefault(m.group(1).upper(),[]).append(p)
if 'QQQ' not in files:raise FileNotFoundError('Missing QQQ 1-minute data')
print('Matched symbols:',len(files),'| source files:',sum(map(len,files.values())))
def daily_bars(paths):
    parts={}
    for p in paths:
        try:
            a=pd.read_csv(p)
            col=next((c for c in a if c.lower() in ('datetime','date','timestamp')),a.columns[0])
            a.index=pd.to_datetime(a[col],utc=True,errors='coerce').dt.tz_convert('America/New_York')
            a=a[['Open','High','Low','Close']].apply(pd.to_numeric,errors='coerce').dropna()
            a=a.between_time('09:30','15:59')
            for day,g in a.groupby(a.index.date):
                parts.setdefault(str(day),[]).append(g)
        except Exception as exc: print('Skipped',p.name,str(exc)[:100])
    result={}
    for date,group in parts.items():
        a=pd.concat(group).sort_index()
        a=a[~a.index.duplicated(keep='last')]
        start=pd.Timestamp(date+' 09:30',tz='America/New_York')
        end=pd.Timestamp(date+' 15:59',tz='America/New_York')
        if start not in a.index or end not in a.index or len(a)<350:continue
        op=float(a.loc[start,'Open'])
        if op<=0:continue
        result[date]=dict(open=op,high=float(a.High.max()),low=float(a.Low.min()),
                          close=float(a.loc[end,'Close']),one_minute_bars=len(a))
    return result
qqq_days=daily_bars(files['QQQ'])
market_dates=sorted(qqq_days)
if len(market_dates)<11:raise RuntimeError('Fewer than eleven complete QQQ sessions')
print('Complete QQQ market dates:',len(market_dates),'| from:',market_dates[0],'through:',market_dates[-1])

In [ ]:
daily=[]; adr=[]
for n,ticker in enumerate(sorted(set(primary.ticker)),1):
    if ticker not in files:continue
    dates=daily_bars(files[ticker])
    for date,bar in dates.items():
        daily.append(dict(ticker=ticker,date=date,**bar,
                          daily_range_pct=100*(bar['high']-bar['low'])/bar['open']))
    s=pd.Series({d:100*(x['high']-x['low'])/x['open'] for d,x in dates.items()},dtype=float).reindex(market_dates)
    hist=s.rolling(10,min_periods=10).mean().shift(1)
    for date in set(primary.loc[primary.ticker.eq(ticker),'date']):
        if date in hist.index and pd.notna(hist.loc[date]):
            adr.append(dict(ticker=ticker,date=date,adr10_prior_pct=float(hist.loc[date])))
    if n%100==0:print('Calculated historical ADR for',n,'tickers')
daily=pd.DataFrame(daily); adr=pd.DataFrame(adr,columns=['ticker','date','adr10_prior_pct'])
daily.to_csv(OUT/'completed_session_ranges.csv',index=False)
adr.to_csv(OUT/'prior_10_day_adr_ticker_dates.csv',index=False)
a=primary.merge(adr,on=['ticker','date'],how='left',validate='many_to_one')
a['adr_band']=pd.cut(a.adr10_prior_pct,[-np.inf,3,4,5,7,np.inf],
                     right=False,labels=['<3%','3–<4%','4–<5%','5–<7%','≥7%']).astype(str)
a.loc[a.adr10_prior_pct.isna(),'adr_band']='Unknown: fewer than 10 complete prior sessions'
a['exclude_three_oct']=a.date.isin(['2026-10-12','2026-10-14','2026-10-16'])
a['exclude_nov20']=a.date.eq('2026-11-20')
a.to_csv(OUT/'short_trades_with_prior_adr.csv',index=False)
print('ADR coverage:',a.adr10_prior_pct.notna().sum(),'/',len(a),
      '| known tickers:',a.loc[a.adr10_prior_pct.notna(),'ticker'].nunique(),
      '| known dates:',a.loc[a.adr10_prior_pct.notna(),'date'].nunique())
display(a.groupby('adr_band',dropna=False).agg(trades=('R','size'),tickers=('ticker','nunique'),dates=('date','nunique')).reset_index())

In [ ]:
def summary(frame,by):
    if frame.empty:return pd.DataFrame()
    v=frame.assign(win=frame.result.eq('win').astype(int),
                   loss=frame.result.eq('loss').astype(int),
                   neutral=frame.result.eq('neutral').astype(int))
    t=v.groupby(by,dropna=False,observed=True).agg(
       trades=('R','size'),wins=('win','sum'),losses=('loss','sum'),
       neutrals=('neutral','sum'),net_R=('R','sum'),mean_R=('R','mean'),
       dates=('date','nunique'),tickers=('ticker','nunique')).reset_index()
    assert (t.wins+t.losses+t.neutrals==t.trades).all()
    return t.round({'net_R':3,'mean_R':4})
def save(name,frame,by,preview=80):
    t=summary(frame,by);t.to_csv(OUT/(name+'.csv'),index=False)
    print('\n'+name);display(t.head(preview))
known=a[a.adr10_prior_pct.notna()]
save('coverage_known_vs_missing',a,['adr_band'])
save('adr_overall',known,['adr_band'])
save('adr_four_way',known,['adr_band','entry_window','qqq_vs_trade'])
save('adr_four_way_history',known,['adr_band','history_group','entry_window','qqq_vs_trade'])
save('adr_four_way_price',known,['adr_band','price_group','entry_window','qqq_vs_trade'])
save('adr_by_date',known,['adr_band','date','entry_window','qqq_vs_trade'],150)
save('adr_without_three_oct',known[~known.exclude_three_oct],['adr_band','entry_window','qqq_vs_trade'])
save('adr_without_nov20',known[~known.exclude_nov20],['adr_band','entry_window','qqq_vs_trade'])
save('adr_without_both',known[~known.exclude_nov20&~known.exclude_three_oct],['adr_band','entry_window','qqq_vs_trade'])
save('adr_by_ticker',known,['adr_band','ticker','entry_window','qqq_vs_trade'],50)
print('Results saved in:',OUT)